# PKG network probe (F3, step 0): what the cluster can run, and how big each view of the graph is

**PNC Treasury Management · DSI · Payment Knowledge Graph** · read-only: writes work files (deleted at the end) and
CSVs in `OUT_DIR`, no tables.

Answers the questions the F3 design depends on:

| § | question |
|---|---|
| §1 | which graph libraries are installed (GraphFrames jar, igraph, leidenalg, NetworKit, cuGraph, PyG, DGL), is there a GPU on the notebook host or the executors, how much memory the notebook host has; a small Leiden timing test if a library is there |
| §2 | **internal view** (customer ↔ customer, on-us): edges, reciprocity, which customers have any on-us edge, degree tails, internal share of dollars |
| §3 | **observed view** (customers + outside entities): entity degree (PNC customers per entity), how much is degree 1, by name class and hub tier; customers' counterparty counts |
| §4 | **projected view** (scope customers linked through a shared counterparty): pair count for each hub cut, customers covered, distinct pairs and projected degree from a sample |
| §5 | edge persistence (months active in the window) and dollar tails, to choose edge weights |
| §6 | summary and clean-up |

Window: the last `WINDOW_MONTHS` published edge months. Own accounts (`is_self`) left out. Scope customers (nodes of the
projected view) = TM customers and organisations. Counterparty ids are mapped to entities (`ent_l2`, lowest per id).
Light enough to run next to another job on the "acc" profile.

## Settings

In [ ]:
# ---- Settings ------------------------------------------------------------------------------------
DB = "bdahd01p_dlcdi1_cdi_tm"
REF = "_v5"
T_EDGE_M = f"{DB}.pkg_edge_monthly"
T_CUST   = f"{DB}.pkg_cust_dim{REF}"
T_MAP    = f"{DB}.pkg_cpty_entity_map{REF}"
T_ENT    = f"{DB}.pkg_cpty_entity_dim{REF}"
T_CPM    = f"{DB}.pkg_cpty_metrics_monthly{REF}"
OUT_DIR  = "../metrics/network/probe"

WINDOW_MONTHS = 3              # trailing window ending at LAST_MONTH
LAST_MONTH    = None           # None = latest published edge month
CAPS = [20, 50, 100, 200, 500, 1000, 5000, None]   # projection: keep a shared counterparty with at most this many
                                                   # scope customers (None = no cap)
SAMPLE_CAPS    = [100, 500]    # caps for the distinct-pair estimate (sampled scope customers)
SAMPLE_PCT     = 1.0           # % of scope customers sampled (lowered automatically to keep the join small)
SAMPLE_ROWS_MAX = 100_000_000  # target rows of the sampled pair join
LEIDEN_TEST = (200_000, 1_000_000)   # nodes, edges of the random-graph timing test
KEEP_WORK = False              # keep the window pair files (int, ce) for the design step

# ---- cluster (no effect on the numbers)
SHUFFLE_PARTITIONS = 2000
RESOURCE_PROFILE = "acc"
RESOURCES = {
    "acc":      {"spark.executor.memory": "4g", "spark.executor.memoryOverhead": "1g", "spark.executor.cores": "1",
                 "spark.driver.memory": "4g", "spark.yarn.am.memory": "4g", "spark.yarn.am.memoryOverhead": "1g",
                 "spark.dynamicAllocation.maxExecutors": "512", "spark.driver.maxResultSize": "2g"},
    "standard": {"spark.executor.memory": "8g", "spark.executor.memoryOverhead": "3g", "spark.executor.cores": "2",
                 "spark.driver.memory": "4g", "spark.yarn.am.memory": "4g", "spark.yarn.am.memoryOverhead": "1g",
                 "spark.dynamicAllocation.maxExecutors": "512", "spark.driver.maxResultSize": "4g"},
}
HDFS_REPLICATION = 2

## Setup

In [ ]:
# ---- Setup ---------------------------------------------------------------------------------------
import numpy as np
import pandas as pd
import warnings
for _alias, _target in {"object0": np.object_, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp}.items():
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
if "bool" not in np.__dict__:
    np.bool = np.bool_
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
warnings.filterwarnings("ignore")

import os, sys, math, time, json, platform, shutil, subprocess, importlib, importlib.util, resource, glob
import datetime as dt, html as _html
from IPython.display import display, HTML
from pyspark.sql import SparkSession, Window, functions as F

_b = SparkSession.builder.appName("pkg_network_probe").config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
for _k, _v in RESOURCES[RESOURCE_PROFILE].items():
    _b = _b.config(_k, _v)
_b = (_b.config("spark.hadoop.dfs.replication", str(HDFS_REPLICATION))
        .config("spark.dynamicAllocation.enabled", "true")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .config("spark.sql.execution.arrow.pyspark.enabled", "true")
        .config("spark.sql.adaptive.enabled", "true").config("spark.sql.adaptive.skewJoin.enabled", "true")
        .config("spark.ui.showConsoleProgress", "false")
        .config("spark.ui.retainedJobs", "100").config("spark.ui.retainedStages", "100")
        .config("spark.ui.retainedTasks", "10000").config("spark.sql.ui.retainedExecutions", "20")
        .enableHiveSupport())
spark = _b.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
spark.sparkContext.setLogLevel("ERROR")
for _n in ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage", "org.apache.kafka",
           "org.apache.spark.network", "io.netty"):
    try:
        _L = spark._jvm.org.apache.logging.log4j
        _L.core.config.Configurator.setLevel(_n, _L.Level.OFF)
    except Exception:
        pass
HCONF = spark._jsc.hadoopConfiguration()
HCONF.set("dfs.replication", str(HDFS_REPLICATION))
pd.set_option("display.max_rows", 300); pd.set_option("display.max_columns", 80)

def tbl(pdf, pct=(), dec=()):
    pdf = pdf.copy()
    fmt = {}
    for c in pdf.columns:
        if c in pct:                                   fmt[c] = "{:.1%}"
        elif c in dec:                                 fmt[c] = "{:,.2f}"
        elif pd.api.types.is_bool_dtype(pdf[c]):       continue
        elif pd.api.types.is_integer_dtype(pdf[c]):    fmt[c] = "{:,}"
        elif pd.api.types.is_float_dtype(pdf[c]):      fmt[c] = "{:,.1f}"
    sty = pdf.style.format(fmt, na_rep="—")
    try:
        sty = sty.hide(axis="index")
    except AttributeError:
        sty = sty.hide_index()
    sty = sty.set_table_styles([{"selector": "th, td", "props": "padding: 2px 8px; font-size: 12px;"}])
    return sty.to_html()
def sec(title, body=""):
    return f"<h4 style='margin:14px 0 4px 0'>{_html.escape(title)}</h4>{body}"
def para(text, color=None):
    st = f"; color:{color}" if color else ""
    return f"<p style='margin:4px 0; font-size:13px{st}'>{_html.escape(text)}</p>"
def out(*parts):
    display(HTML("<div>" + "".join(p for p in parts if p) + "</div>"))

JPath = spark._jvm.org.apache.hadoop.fs.Path
def fs_for(p):
    return JPath(p).getFileSystem(HCONF)
def exists(p):
    return fs_for(p).exists(JPath(p))
def delete(p):
    return fs_for(p).delete(JPath(p), True) if exists(p) else False
def size_gb(p):
    return fs_for(p).getContentSummary(JPath(p)).getLength() / 1e9 if exists(p) else 0.0
def table_location(t):
    return [r[1] for r in spark.sql(f"DESCRIBE TABLE EXTENDED {t}").collect() if r[0] == "Location"][0].rstrip("/")
def month_add(m, k):
    t = int(m[:4]) * 12 + int(m[5:7]) - 1 + k
    return f"{t // 12:04d}-{t % 12 + 1:02d}"
def need(table, cols):
    dt_ = dict(spark.table(table).dtypes)
    miss = [c for c in cols if c not in dt_]
    assert not miss, f"{table}: missing {miss}; has {sorted(dt_)}"
    return dt_
def partitions(table, key):
    vals = set()
    try:
        for r in spark.sql(f"SHOW PARTITIONS {table}").collect():
            for kv in r[0].split("/"):
                if kv.startswith(key + "="):
                    vals.add(kv.split("=", 1)[1])
    except Exception:
        vals = {str(r[0]) for r in spark.table(table).select(key).distinct().collect()}
    return sorted(v for v in vals if v and v != "None")
def save(pdf, name):
    pdf.to_csv(f"{OUT_DIR}/{name}.csv", index=False)
def bucket(c, edges=(1, 2, 5, 20, 100, 1000, 10000)):
    """Degree buckets 1 · 2–4 · 5–19 · 20–99 · 100–999 · 1,000–9,999 · 10,000+ → (column expression, labels in order)."""
    labels = []
    for i, lo in enumerate(edges):
        hi = edges[i + 1] if i + 1 < len(edges) else None
        labels.append(f"{lo:,}" if hi == lo + 1 else (f"{lo:,}–{hi - 1:,}" if hi else f"{lo:,}+"))
    expr = F.lit(None).cast("string")
    for lo, lab in zip(edges, labels):              # later (higher) buckets overwrite earlier ones
        expr = F.when(c >= lo, F.lit(lab)).otherwise(expr)
    return expr, labels
def order_by_labels(pdf, col, labels):
    pdf = pdf.copy()
    pdf["_o"] = pdf[col].map({l: i for i, l in enumerate(labels)})
    return pdf.sort_values("_o").drop(columns="_o")

DB_LOC = [r[1] for r in spark.sql(f"DESCRIBE DATABASE {DB}").collect() if r[0].lower() == "location"][0].rstrip("/")
WORK = f"{DB_LOC}/_tmp_pkg_network_probe"
os.makedirs(OUT_DIR, exist_ok=True)
T_START = time.time()
out(sec(f"Spark {spark.version} · profile {RESOURCE_PROFILE} · Python {platform.python_version()} · pandas {pd.__version__}"))

## §1 Environment: libraries, GPU, memory

Python packages are checked in the notebook's own environment (where a single-machine Leiden would run). The
GraphFrames check looks for its Java classes in this Spark session (the Python wrapper alone is not enough). Executors
are asked for their host name, CPU count and whether `nvidia-smi` exists. If igraph is installed, a Leiden run on a
random graph (`LEIDEN_TEST`) gives a time and memory reading.

In [ ]:
# §1  Environment
t0 = time.time()
PKGS = ["graphframes", "igraph", "leidenalg", "networkit", "networkx", "cugraph", "cudf", "cupy", "torch",
        "torch_geometric", "dgl", "scipy", "sklearn", "lightgbm", "xgboost", "pyarrow", "numba"]
pk = []
for p in PKGS:
    if importlib.util.find_spec(p) is None:
        pk.append({"package": p, "status": "not installed", "version": ""}); continue
    try:
        m = importlib.import_module(p)
        pk.append({"package": p, "status": "ok", "version": str(getattr(m, "__version__", "?"))})
    except Exception as e:
        pk.append({"package": p, "status": "import fails", "version": str(e).splitlines()[0][:90]})
PK = pd.DataFrame(pk)
HAVE = set(PK[PK.status == "ok"].package)

def jvm_has(cls):
    try:
        parts = cls.split(".")
        o = spark._jvm
        for x in parts:
            o = getattr(o, x)
        return type(o).__name__ == "JavaClass"
    except Exception:
        return False
_sh = os.environ.get("SPARK_HOME", "")
_gf_jars = sorted(os.path.basename(x) for x in glob.glob(f"{_sh}/jars/*graphframes*")) if _sh else []
_conf = dict(spark.sparkContext.getConf().getAll())
env = [
    {"item": "GraphFrames classes in this session", "value": "yes" if jvm_has("org.graphframes.GraphFrame") else "no"},
    {"item": "GraphFrames jar in SPARK_HOME/jars", "value": ", ".join(_gf_jars) or "none"},
    {"item": "spark.jars / spark.jars.packages", "value": f"{_conf.get('spark.jars', '') or '—'} / {_conf.get('spark.jars.packages', '') or '—'}"},
    {"item": "GraphX classes (Scala only)", "value": "yes" if jvm_has("org.apache.spark.graphx.Graph") else "no"},
    {"item": "executor GPU setting", "value": _conf.get("spark.executor.resource.gpu.amount", "none")},
]
# notebook host
try:
    mi = {}
    for line in open("/proc/meminfo"):
        k, v = line.split(":", 1)
        mi[k] = int(v.split()[0]) / 1024 / 1024
    env.append({"item": "host memory total / available (GB)", "value": f"{mi.get('MemTotal', 0):,.1f} / {mi.get('MemAvailable', 0):,.1f}"})
except Exception as e:
    env.append({"item": "host memory", "value": f"unreadable ({e})"})
_cg = None
for p in ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
    try:
        v = open(p).read().strip()
        if v.isdigit() and int(v) < 2 ** 60:
            _cg = int(v) / 1024 ** 3
        break
    except Exception:
        pass
env.append({"item": "container memory limit (GB)", "value": f"{_cg:,.1f}" if _cg else "none found"})
env.append({"item": "host CPUs", "value": str(os.cpu_count())})
_soft, _hard = resource.getrlimit(resource.RLIMIT_AS)
env.append({"item": "address-space limit", "value": "none" if _soft == resource.RLIM_INFINITY else f"{_soft / 1024 ** 3:,.1f} GB"})
_smi = shutil.which("nvidia-smi")
if _smi:
    try:
        r = subprocess.run([_smi, "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True, timeout=15)
        env.append({"item": "GPU on notebook host", "value": r.stdout.strip().replace("\n", "; ") or r.stderr.strip()[:120]})
    except Exception as e:
        env.append({"item": "GPU on notebook host", "value": f"nvidia-smi failed ({e})"})
else:
    env.append({"item": "GPU on notebook host", "value": "no nvidia-smi"})
if "torch" in HAVE:
    import torch
    env.append({"item": "torch CUDA available / devices", "value": f"{torch.cuda.is_available()} / {torch.cuda.device_count()}"})
# executors
def _probe(_):
    import socket, shutil as _s, os as _o
    yield (socket.gethostname(), bool(_s.which("nvidia-smi")), _o.cpu_count())
try:
    ex = spark.sparkContext.parallelize(range(64), 64).mapPartitions(_probe).distinct().collect()
    env.append({"item": "executor hosts seen / with nvidia-smi", "value": f"{len(ex)} / {sum(1 for e in ex if e[1])}"})
    env.append({"item": "executor host CPUs (min–max)", "value": f"{min(e[2] for e in ex)}–{max(e[2] for e in ex)}" if ex else "—"})
except Exception as e:
    env.append({"item": "executor probe", "value": f"failed: {str(e).splitlines()[0][:120]}"})
ENV = pd.DataFrame(env)

# Leiden timing on a random graph
lt = []
n_, m_ = LEIDEN_TEST
if "igraph" in HAVE:
    import igraph as ig
    rss0 = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024 / 1024
    t_ = time.time(); g = ig.Graph.Erdos_Renyi(n=n_, m=m_); tb = time.time() - t_
    t_ = time.time(); cl = g.community_leiden(objective_function="modularity", n_iterations=2); tl = time.time() - t_
    rss1 = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024 / 1024
    lt.append({"library": f"igraph {ig.__version__} community_leiden", "nodes": n_, "edges": m_, "build_s": round(tb, 1),
               "run_s": round(tl, 1), "communities": len(cl), "peak_rss_gb": round(rss1, 2), "rss_growth_gb": round(rss1 - rss0, 2)})
    del g, cl
if "networkit" in HAVE:
    import networkit as nk
    t_ = time.time(); g = nk.generators.ErdosRenyiGenerator(n_, 2 * m_ / (n_ * (n_ - 1))).generate(); tb = time.time() - t_
    t_ = time.time(); _plm = nk.community.PLM(g, True); _plm.run(); pl = _plm.getPartition(); tl = time.time() - t_
    lt.append({"library": f"networkit {nk.__version__} PLM (Louvain)", "nodes": g.numberOfNodes(), "edges": g.numberOfEdges(),
               "build_s": round(tb, 1), "run_s": round(tl, 1), "communities": pl.numberOfSubsets(),
               "peak_rss_gb": round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024 / 1024, 2), "rss_growth_gb": None})
    del g, pl
LT = pd.DataFrame(lt)
save(PK, "env_packages"); save(ENV, "env_cluster")
out(sec(f"§1 Environment · {(time.time() - t0) / 60:.1f} min"),
    sec("Python packages (notebook environment)"), tbl(PK),
    sec("Spark, host, GPU"), tbl(ENV),
    sec(f"Leiden / Louvain timing on a random graph ({n_:,} nodes, {m_:,} edges)"),
    tbl(LT) if len(LT) else para("No igraph or NetworKit in this environment: no single-machine Leiden / Louvain here."))

## §2 Window and the internal view (customer ↔ customer)

Edges of the window, own accounts out. Internal directed edges (payer customer → payee customer), undirected pairs and
reciprocity; which customers have any on-us edge, by segment; degree tails; the largest internal nodes (PNC's own
accounts and processors show here); internal share of the window's dollars.

In [ ]:
# §2  Window and internal view
t0 = time.time()
need(T_EDGE_M, ["src_id", "src_kind", "dst_id", "dst_kind", "is_self", "amount", "month"])
need(T_CUST, ["mdm_id", "party_type", "is_tm", "is_pnc_own", "naics2"])
need(T_MAP, ["cpty_id", "ent_l2"])
need(T_ENT, ["ent_id", "name_class"])
need(T_CPM, ["ent_id", "hub_tier", "month"])
E_LOC = table_location(T_EDGE_M)
EDGE_MONTHS = [m for m in partitions(T_EDGE_M, "month") if exists(f"{E_LOC}/month={m}/_PKG_BUILD")]
LAST = LAST_MONTH or EDGE_MONTHS[-1]
WIN = [m for m in EDGE_MONTHS if month_add(LAST, -WINDOW_MONTHS) < m <= LAST]
CPM_MONTH = max([m for m in partitions(T_CPM, "month") if m <= LAST], default=None)

cust = (spark.table(T_CUST)
          .select(F.trim(F.col("mdm_id")).alias("mdm_id"), "party_type", "naics2",
                  F.coalesce(F.col("is_tm").cast("boolean"), F.lit(False)).alias("is_tm"),
                  F.coalesce(F.col("is_pnc_own").cast("boolean"), F.lit(False)).alias("is_pnc_own"))
          .withColumn("is_org", F.col("party_type") == "O")
          .withColumn("segment", F.when(F.col("is_tm") & F.col("is_org"), "TM org").when(F.col("is_tm"), "TM non-org")
                                  .when(F.col("is_org"), "non-TM org").otherwise("person / other"))
          .withColumn("in_scope", F.col("is_tm") | F.col("is_org")))
_dup = cust.groupBy("mdm_id").count().where("count > 1").count()
assert _dup == 0, f"{T_CUST}: {_dup} mdm_id with more than one row"
cust.write.mode("overwrite").parquet(f"{WORK}/cust"); cust = spark.read.parquet(f"{WORK}/cust")
SEGS = ["TM org", "TM non-org", "non-TM org", "person / other"]

E = (spark.table(T_EDGE_M).where(F.col("month").isin(WIN) & ~F.coalesce(F.col("is_self").cast("boolean"), F.lit(False)))
       .select(F.trim(F.col("src_id")).alias("src"), "src_kind", F.trim(F.col("dst_id")).alias("dst"), "dst_kind",
               F.col("amount").cast("double").alias("amt"), "month"))
TOT = E.agg(F.sum("amt").alias("usd"), F.count("*").alias("rows")).first()
INT = (E.where((F.col("src_kind") == "CUST") & (F.col("dst_kind") == "CUST") & (F.col("src") != F.col("dst")))
         .groupBy("src", "dst").agg(F.sum("amt").alias("amt"), F.countDistinct("month").alias("months")))
INT.write.mode("overwrite").parquet(f"{WORK}/int"); INT = spark.read.parquet(f"{WORK}/int")
UP = (INT.select(F.least("src", "dst").alias("a"), F.greatest("src", "dst").alias("b"), "amt", "months")
         .groupBy("a", "b").agg(F.sum("amt").alias("amt"), F.max("months").alias("months"), F.count("*").alias("dirs")))
UP.write.mode("overwrite").parquet(f"{WORK}/up"); UP = spark.read.parquet(f"{WORK}/up")
DEGI = (UP.select(F.col("a").alias("mdm_id")).unionAll(UP.select(F.col("b").alias("mdm_id")))
          .groupBy("mdm_id").agg(F.count("*").alias("deg_int")))
DEGI.write.mode("overwrite").parquet(f"{WORK}/degi"); DEGI = spark.read.parquet(f"{WORK}/degi")

ii = INT.agg(F.count("*").alias("n"), F.sum("amt").alias("usd")).first()
uu = UP.agg(F.count("*").alias("n"), F.sum((F.col("dirs") == 2).cast("long")).alias("recip")).first()
sc = cust.select(F.col("mdm_id").alias("a"), F.col("in_scope").alias("sa"))
both = (UP.join(sc, "a", "left").join(sc.withColumnRenamed("a", "b").withColumnRenamed("sa", "sb"), "b", "left")
          .agg(F.sum((F.coalesce("sa", F.lit(False)) & F.coalesce("sb", F.lit(False))).cast("long")).alias("n")).first())
SEGI = (cust.join(DEGI, "mdm_id", "left").groupBy("segment")
          .agg(F.count("*").alias("customers"), F.sum(F.col("deg_int").isNotNull().cast("long")).alias("with_onus_edge"),
               F.expr("percentile_approx(deg_int, 0.5)").alias("deg_p50"), F.expr("percentile_approx(deg_int, 0.9)").alias("deg_p90"),
               F.expr("percentile_approx(deg_int, 0.99)").alias("deg_p99"), F.max("deg_int").alias("deg_max"))
          .toPandas())
SEGI["share_with_onus"] = SEGI.with_onus_edge / SEGI.customers
SEGI = order_by_labels(SEGI, "segment", SEGS)
TOPI = (DEGI.join(cust, "mdm_id", "left").orderBy(F.desc("deg_int")).limit(15)
          .select("mdm_id", "segment", "naics2", "is_pnc_own", "deg_int").toPandas())
SUM_I = pd.DataFrame([
    {"item": "window", "value": f"{WIN[0]} → {WIN[-1]} ({len(WIN)} months)"},
    {"item": "edge rows in window (own accounts out)", "value": f"{TOT['rows']:,}"},
    {"item": "internal directed edges (customer pairs, payer → payee)", "value": f"{ii['n']:,}"},
    {"item": "internal undirected pairs / reciprocal", "value": f"{uu['n']:,} / {uu['recip'] or 0:,} ({(uu['recip'] or 0) / max(uu['n'], 1):.1%})"},
    {"item": "undirected pairs with both ends in scope (TM or organisation)", "value": f"{both['n'] or 0:,}"},
    {"item": "internal share of window dollars", "value": f"{(ii['usd'] or 0) / max(TOT['usd'] or 1, 1):.1%}"},
])
save(SUM_I, "internal_summary"); save(SEGI, "internal_by_segment"); save(TOPI, "internal_top_degree")
out(sec(f"§2 Internal view · {(time.time() - t0) / 60:.1f} min"), tbl(SUM_I),
    sec("Customers with any on-us edge, by segment (degree = distinct on-us partners)"),
    tbl(SEGI[["segment", "customers", "with_onus_edge", "share_with_onus", "deg_p50", "deg_p90", "deg_p99", "deg_max"]],
        pct=("share_with_onus",)),
    sec("Largest internal nodes"), tbl(TOPI))

## §3 Observed view (customers + outside entities)

One row per customer × entity in the window. Entity degree = distinct PNC customers it trades with (and how many of
them are scope customers). An entity of degree 1 carries no structure for global algorithms (it stays as dollars on
its customer). Split by `name_class` and by `hub_tier` (counterparty metrics, latest month ≤ the window end).
Unmapped counterparty ids are counted as their own entity.

In [ ]:
# §3  Observed view
t0 = time.time()
mp = (spark.table(T_MAP).select(F.trim(F.col("cpty_id")).alias("cpty"), F.col("ent_l2").cast("string").alias("ent"))
        .groupBy("cpty").agg(F.min("ent").alias("ent")))
a_ = E.where((F.col("src_kind") == "CUST") & (F.col("dst_kind") == "CPTY")).select(
    F.col("src").alias("cust"), F.col("dst").alias("cpty"), F.lit("out").alias("dir"), "amt", "month")
b_ = E.where((F.col("src_kind") == "CPTY") & (F.col("dst_kind") == "CUST")).select(
    F.col("dst").alias("cust"), F.col("src").alias("cpty"), F.lit("in").alias("dir"), "amt", "month")
CE = (a_.unionByName(b_).join(mp, "cpty", "left")
        .withColumn("mapped", F.col("ent").isNotNull().cast("int"))
        .withColumn("ent", F.coalesce(F.col("ent"), F.concat(F.lit("U:"), F.col("cpty"))))
        .groupBy("cust", "ent")
        .agg(F.sum(F.when(F.col("dir") == "in", F.col("amt")).otherwise(0.0)).alias("amt_in"),
             F.sum(F.when(F.col("dir") == "out", F.col("amt")).otherwise(0.0)).alias("amt_out"),
             F.countDistinct("month").alias("months"), F.max("mapped").alias("mapped"))
        .join(cust.select(F.col("mdm_id").alias("cust"), "in_scope", "segment"), "cust", "left")
        .withColumn("in_scope", F.coalesce("in_scope", F.lit(False))))
CE.write.mode("overwrite").parquet(f"{WORK}/ce"); CE = spark.read.parquet(f"{WORK}/ce")

ent_attr = spark.table(T_ENT).select(F.col("ent_id").cast("string").alias("ent"), "name_class")
hub_attr = (spark.table(T_CPM).where(F.col("month") == CPM_MONTH).select(F.col("ent_id").cast("string").alias("ent"), "hub_tier")
              .groupBy("ent").agg(F.max("hub_tier").alias("hub_tier"))) if CPM_MONTH else None
ENTD = (CE.groupBy("ent").agg(F.count("*").alias("deg"), F.sum(F.col("in_scope").cast("long")).alias("deg_scope"),
                              F.sum(F.col("amt_in") + F.col("amt_out")).alias("amt"), F.max("mapped").alias("mapped"))
          .join(ent_attr, "ent", "left"))
if hub_attr is not None:
    ENTD = ENTD.join(hub_attr, "ent", "left")
else:
    ENTD = ENTD.withColumn("hub_tier", F.lit(None).cast("int"))
ENTD.write.mode("overwrite").parquet(f"{WORK}/entd"); ENTD = spark.read.parquet(f"{WORK}/entd")

bx, BL = bucket(F.col("deg"))
DB_ = (ENTD.groupBy(bx.alias("degree")).agg(F.count("*").alias("entities"), F.sum("deg").alias("pairs"), F.sum("amt").alias("usd"))
           .toPandas())
DB_["share_entities"] = DB_.entities / DB_.entities.sum(); DB_["share_pairs"] = DB_.pairs / DB_.pairs.sum()
DB_["share_usd"] = DB_.usd / DB_.usd.sum()
DB_ = order_by_labels(DB_, "degree", BL)
bs, _ = bucket(F.col("deg_scope"))
DS_ = (ENTD.where(F.col("deg_scope") >= 1).groupBy(bs.alias("scope_degree"))
           .agg(F.count("*").alias("entities"), F.sum("deg_scope").alias("scope_pairs")).toPandas())
DS_["share_entities"] = DS_.entities / DS_.entities.sum()
DS_ = order_by_labels(DS_, "scope_degree", BL)
NC_ = (ENTD.groupBy(F.coalesce(F.col("name_class"), F.when(F.col("mapped") == 0, "(unmapped id)").otherwise("(no class)")).alias("name_class"))
           .agg(F.count("*").alias("entities"), F.sum((F.col("deg") >= 2).cast("long")).alias("deg_ge2"),
                F.sum((F.col("deg_scope") >= 2).cast("long")).alias("scope_deg_ge2"), F.sum("deg").alias("pairs"),
                F.sum("amt").alias("usd"))
           .toPandas().sort_values("entities", ascending=False))
NC_["share_usd"] = NC_.usd / NC_.usd.sum()
HT_ = (ENTD.groupBy(F.coalesce(F.col("hub_tier").cast("string"), F.lit("(none)")).alias("hub_tier"))
           .agg(F.count("*").alias("entities"), F.min("deg").alias("deg_min"), F.expr("percentile_approx(deg, 0.5)").alias("deg_p50"),
                F.max("deg").alias("deg_max"), F.sum("deg").alias("pairs"))
           .toPandas().sort_values("hub_tier"))
CD = CE.groupBy("cust", "segment").agg(F.count("*").alias("n_ent"))
CDQ = (CD.groupBy("segment").agg(F.count("*").alias("customers"), F.expr("percentile_approx(n_ent, 0.5)").alias("p50"),
                                 F.expr("percentile_approx(n_ent, 0.9)").alias("p90"), F.expr("percentile_approx(n_ent, 0.99)").alias("p99"),
                                 F.max("n_ent").alias("max"), F.sum("n_ent").alias("pairs")).toPandas())
CDQ = order_by_labels(CDQ, "segment", SEGS)
TOPC = (CD.orderBy(F.desc("n_ent")).limit(15).join(cust.select(F.col("mdm_id").alias("cust"), "naics2", "is_pnc_own"), "cust", "left")
          .orderBy(F.desc("n_ent")).toPandas())
TOPE = (ENTD.orderBy(F.desc("deg")).limit(15).select("ent", "name_class", "hub_tier", "deg", "deg_scope").toPandas())
ee = ENTD.agg(F.count("*").alias("n"), F.sum((F.col("deg") >= 2).cast("long")).alias("ge2"),
              F.sum(F.when(F.col("deg") >= 2, F.col("deg")).otherwise(0)).alias("pairs_ge2"), F.sum("deg").alias("pairs"),
              F.sum((F.col("mapped") == 0).cast("long")).alias("unmapped")).first()
SUM_O = pd.DataFrame([
    {"item": "entities in window", "value": f"{ee['n']:,} (unmapped ids {ee['unmapped']:,})"},
    {"item": "customer × entity pairs", "value": f"{ee['pairs']:,}"},
    {"item": "entities with ≥ 2 PNC customers (enter global algorithms)", "value": f"{ee['ge2']:,} ({ee['ge2'] / max(ee['n'], 1):.1%}), {ee['pairs_ge2']:,} pairs"},
    {"item": "hub_tier month", "value": CPM_MONTH or "none"},
])
for n_, d_ in [("observed_summary", SUM_O), ("observed_entity_degree", DB_), ("observed_entity_scope_degree", DS_),
               ("observed_name_class", NC_), ("observed_hub_tier", HT_), ("observed_customer_degree", CDQ),
               ("observed_top_customers", TOPC), ("observed_top_entities", TOPE)]:
    save(d_, n_)
out(sec(f"§3 Observed view · {(time.time() - t0) / 60:.1f} min"), tbl(SUM_O),
    sec("Entities by degree (distinct PNC customers)"),
    tbl(DB_[["degree", "entities", "share_entities", "pairs", "share_pairs", "share_usd"]], pct=("share_entities", "share_pairs", "share_usd")),
    sec("Entities by scope degree (distinct TM / organisation customers)"),
    tbl(DS_[["scope_degree", "entities", "share_entities", "scope_pairs"]], pct=("share_entities",)),
    sec("By name class"), tbl(NC_[["name_class", "entities", "deg_ge2", "scope_deg_ge2", "pairs", "share_usd"]], pct=("share_usd",)),
    sec("By hub tier"), tbl(HT_),
    sec("Customers: number of outside entities, by segment"), tbl(CDQ),
    sec("Largest entities"), tbl(TOPE),
    sec("Customers with the most outside entities"), tbl(TOPC))

## §4 Projected view (scope customers linked through a shared counterparty)

Middle nodes: outside entities and customers (a customer that two scope customers both trade with). Members: scope
customers. A middle node with d members creates d(d − 1) / 2 member pairs. For each cap (middle nodes with more members
are dropped) and hub rule: middle nodes kept, pair rows (an upper bound: two customers sharing several counterparties
count once per shared counterparty), and scope customers with at least one projected link. A sample of scope customers
gives the ratio of distinct partners to pair rows (so the distinct pair count) and the projected degree.

In [ ]:
# §4  Projected view
t0 = time.time()
CE = spark.read.parquet(f"{WORK}/ce"); UP = spark.read.parquet(f"{WORK}/up"); ENTD = spark.read.parquet(f"{WORK}/entd")
scope_ids = cust.where("in_scope").select(F.col("mdm_id").alias("member"))
Mi = CE.where("in_scope").select(F.concat(F.lit("E:"), F.col("ent")).alias("mid"), F.col("cust").alias("member"))
Mc = (UP.select(F.col("a").alias("mid"), F.col("b").alias("member"))
        .unionByName(UP.select(F.col("b").alias("mid"), F.col("a").alias("member")))
        .join(scope_ids, "member", "left_semi").withColumn("mid", F.concat(F.lit("C:"), F.col("mid"))))
M = Mi.unionByName(Mc)
M.write.mode("overwrite").parquet(f"{WORK}/m"); M = spark.read.parquet(f"{WORK}/m")
DM = (M.groupBy("mid").agg(F.count("*").alias("d"))
        .join(ENTD.select(F.concat(F.lit("E:"), F.col("ent")).alias("mid"), "hub_tier"), "mid", "left")
        .withColumn("kind", F.substring("mid", 1, 1))
        .withColumn("ht", F.coalesce(F.col("hub_tier").cast("int"), F.lit(-1))))
DM.write.mode("overwrite").parquet(f"{WORK}/dm"); DM = spark.read.parquet(f"{WORK}/dm")
G = DM.where("d >= 2").groupBy("d", "ht", "kind").agg(F.count("*").alias("mids")).toPandas()
G["pairs"] = G.mids.astype(float) * G.d.astype(float) * (G.d.astype(float) - 1) / 2
G["rows"] = G.mids.astype(float) * G.d.astype(float)
RULES = {"all middle nodes": lambda g: g, "no hub_tier 3": lambda g: g[g.ht < 3], "no hub_tier ≥ 2": lambda g: g[g.ht < 2]}
COND = {"all middle nodes": F.lit(True), "no hub_tier 3": F.col("ht") < 3, "no hub_tier ≥ 2": F.col("ht") < 2}
n_scope = scope_ids.count()
# members covered: smallest d among a member's middle nodes (d ≥ 2), per hub rule
cov = (M.join(DM.where("d >= 2"), "mid")
         .groupBy("member").agg(*[F.min(F.when(c, F.col("d"))).alias(f"m{i}") for i, c in enumerate(COND.values())]))
cov.write.mode("overwrite").parquet(f"{WORK}/cov"); cov = spark.read.parquet(f"{WORK}/cov")
aggs = []
for i in range(len(COND)):
    for cap in CAPS:
        cnd = F.col(f"m{i}").isNotNull() if cap is None else (F.col(f"m{i}") <= cap)
        aggs.append(F.sum(cnd.cast("long")).alias(f"c{i}_{cap}"))
covr = cov.agg(*aggs).first().asDict()
rows = []
for i, (rn, rf) in enumerate(RULES.items()):
    g = rf(G)
    for cap in CAPS:
        gg = g if cap is None else g[g.d <= cap]
        rows.append({"hub_rule": rn, "cap": "none" if cap is None else f"{cap:,}", "middle_nodes": int(gg.mids.sum()),
                     "of_which_entities": int(gg[gg.kind == "E"].mids.sum()), "member_rows": int(gg.rows.sum()),
                     "pair_rows_bn": gg.pairs.sum() / 1e9, "pair_rows_via_entities_bn": gg[gg.kind == "E"].pairs.sum() / 1e9,
                     "customers_covered": int(covr[f"c{i}_{cap}"] or 0), "share_of_scope": (covr[f"c{i}_{cap}"] or 0) / max(n_scope, 1)})
PR = pd.DataFrame(rows)

# distinct partners from a sample (rule: no hub_tier 3)
smp = []
for cap in SAMPLE_CAPS:
    tot = float(G[(G.ht < 3) & (G.d <= cap)].pairs.sum())
    frac = min(SAMPLE_PCT / 100, SAMPLE_ROWS_MAX / max(2 * tot, 1.0))
    k = max(1, int(round(frac * 1_000_000)))
    keep = DM.where((F.col("d") >= 2) & (F.col("d") <= cap) & (F.col("ht") < 3)).select("mid")
    Mk = M.join(keep, "mid", "left_semi")
    S = Mk.where(F.abs(F.xxhash64("member")) % 1_000_000 < k)
    R = (S.alias("s").join(Mk.alias("p"), "mid").where(F.col("s.member") != F.col("p.member"))
           .select(F.col("s.member").alias("member"), F.col("p.member").alias("partner")))
    per = R.groupBy("member").agg(F.count("*").alias("rows"), F.countDistinct("partner").alias("partners"))
    q = per.agg(F.count("*").alias("n"), F.sum("rows").alias("rows"), F.sum("partners").alias("partners"),
                F.expr("percentile_approx(partners, 0.5)").alias("p50"), F.expr("percentile_approx(partners, 0.9)").alias("p90"),
                F.expr("percentile_approx(partners, 0.99)").alias("p99"), F.max("partners").alias("max")).first()
    ratio = (q["partners"] or 0) / max(q["rows"] or 1, 1)
    smp.append({"cap": f"{cap:,}", "sampled_customers_with_links": q["n"], "sample_share": k / 1_000_000,
                "partners_per_pair_row": ratio, "est_distinct_pairs_m": tot * ratio / 1e6,
                "proj_degree_p50": q["p50"], "proj_degree_p90": q["p90"], "proj_degree_p99": q["p99"], "proj_degree_max": q["max"]})
SMP = pd.DataFrame(smp)
save(PR, "projection_by_cap"); save(SMP, "projection_sample"); save(G, "projection_middle_degree")
out(sec(f"§4 Projected view · {n_scope:,} scope customers · {(time.time() - t0) / 60:.1f} min"),
    sec("Pair rows by cap and hub rule (upper bound on distinct pairs)"),
    tbl(PR, pct=("share_of_scope",), dec=("pair_rows_bn", "pair_rows_via_entities_bn")),
    sec("Sample (rule: no hub_tier 3): distinct partners per pair row, estimated distinct pairs, projected degree"),
    tbl(SMP, pct=("sample_share",), dec=("partners_per_pair_row", "est_distinct_pairs_m")))

## §5 Edge persistence and dollar tails (choice of edge weights)

Months active per pair in the window and the dollars on each; dollar quantiles per pair; share of the window's dollars
on the largest 0.01% and 0.1% of pairs (exact top-k).

In [ ]:
# §5  Persistence and dollar tails
t0 = time.time()
CE = spark.read.parquet(f"{WORK}/ce").withColumn("usd", F.col("amt_in") + F.col("amt_out"))
INT = spark.read.parquet(f"{WORK}/int").withColumnRenamed("amt", "usd")
def persistence(df, name):
    p = df.groupBy("months").agg(F.count("*").alias("pairs"), F.sum("usd").alias("usd")).toPandas().sort_values("months")
    p["share_pairs"] = p.pairs / p.pairs.sum(); p["share_usd"] = p.usd / p.usd.sum()
    p.insert(0, "view", name)
    return p
PS = pd.concat([persistence(CE, "customer × entity"), persistence(INT, "internal (directed)")])
def tails(df, name):
    n = df.count()
    tot = df.agg(F.sum("usd")).first()[0] or 1.0
    q = df.approxQuantile("usd", [0.5, 0.9, 0.99, 0.999], 0.0001)
    r = {"view": name, "pairs": n, "p50": q[0], "p90": q[1], "p99": q[2], "p99_9": q[3]}
    for share in (0.0001, 0.001):
        k = max(1, int(math.ceil(share * n)))
        top = df.orderBy(F.desc("usd")).limit(k).agg(F.sum("usd")).first()[0] or 0.0
        r[f"usd_share_top_{share:.2%}"] = top / tot
    r["max"] = df.agg(F.max("usd")).first()[0]
    return r
TL = pd.DataFrame([tails(CE, "customer × entity"), tails(INT, "internal (directed)")])
save(PS, "persistence"); save(TL, "dollar_tails")
out(sec(f"§5 Persistence and dollar tails · {(time.time() - t0) / 60:.1f} min"),
    sec(f"Months active per pair (window of {len(WIN)} months)"), tbl(PS, pct=("share_pairs", "share_usd")),
    sec("Dollars per pair over the window"),
    tbl(TL, pct=tuple(c for c in TL.columns if c.startswith("usd_share"))))

## §6 Summary and clean-up

In [ ]:
# §6  Summary and clean-up
_w = size_gb(WORK)
if not KEEP_WORK:
    delete(WORK)
r100 = PR[(PR.hub_rule == "no hub_tier 3") & (PR.cap == "100")]
s100 = SMP[SMP.cap == "100"]
SUMM = pd.DataFrame([
    {"item": "GraphFrames usable", "value": ENV.loc[ENV.item == "GraphFrames classes in this session", "value"].iloc[0]},
    {"item": "single-machine community detection", "value": ", ".join(sorted(HAVE & {"igraph", "leidenalg", "networkit"})) or "none"},
    {"item": "GPU libraries", "value": ", ".join(sorted(HAVE & {"cugraph", "cudf", "torch", "torch_geometric", "dgl"})) or "none"},
    {"item": "scope customers with any on-us edge",
     "value": f"{int(SEGI[SEGI.segment.isin(['TM org', 'TM non-org', 'non-TM org'])].with_onus_edge.sum()):,} of "
              f"{int(SEGI[SEGI.segment.isin(['TM org', 'TM non-org', 'non-TM org'])].customers.sum()):,}"},
    {"item": "entities with ≥ 2 PNC customers", "value": f"{ee['ge2']:,} of {ee['n']:,}"},
    {"item": "projection, cap 100, no hub_tier 3: pair rows / est. distinct pairs / customers covered",
     "value": (f"{r100.pair_rows_bn.iloc[0]:,.2f} bn / {s100.est_distinct_pairs_m.iloc[0]:,.0f} m / "
               f"{int(r100.customers_covered.iloc[0]):,} ({r100.share_of_scope.iloc[0]:.1%})") if len(r100) and len(s100) else "—"},
    {"item": "work files", "value": f"{_w:,.1f} GB " + ("kept" if KEEP_WORK else "deleted")},
    {"item": "CSV files", "value": OUT_DIR},
    {"item": "run time", "value": f"{(time.time() - T_START) / 60:.1f} min"},
])
save(SUMM, "summary")
out(sec("§6 Summary"), tbl(SUMM))